# <center>  **<span style="font-size:80px;">Físicos</span>** </center>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..")))
from src.config import DatasetKeys
from src.config import Paths
Paths.init_project()

In [ ]:
df = pd.read_csv(Paths.PROC_CSV_AMAEM)

**<center><span style="font-size:40px;">Modelo Predictivo (Tendencia + Fourier)</span></center>**

In [ ]:
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:

# 1. CARGA Y PREPARACIÓN DE DATOS TEMPORALES
print("1. Agrupando el consumo de toda la ciudad por meses...")

# Asumimos que el dataframe 'df' ya está cargado en tu notebook. 
# Si no, descomenta la siguiente línea:
# df = pd.read_csv(Paths.DATA_DIR / 'AMAEM (1).csv')

# Nombres de columnas según tu CSV
col_fecha = 'fecha'
col_consumo = 'consumo'

# Aseguramos que la fecha es tipo datetime
df_temporal = df.copy()
df_temporal[col_fecha] = pd.to_datetime(df_temporal[col_fecha])

# Aseguramos que el consumo es numérico (limpiando comas si las hubiera)
if df_temporal[col_consumo].dtype == 'O': # Si es object/string
    df_temporal[col_consumo] = pd.to_numeric(df_temporal[col_consumo].astype(str).str.replace(',', ''), errors='coerce')

# Agrupamos todo el consumo de Alicante por mes
df_ts = df_temporal.groupby(col_fecha)[col_consumo].sum().reset_index()
df_ts = df_ts.sort_values(col_fecha)

# Variables para la física
meses_fechas = df_ts[col_fecha].dt.strftime('%Y-%m').values
consumo_real = df_ts[col_consumo].values
meses_historicos = np.arange(len(consumo_real)) # Eje X [0, 1, 2...]

# 2. DEFINICIÓN DEL MODELO MATEMÁTICO
print("2. Configurando Ecuación (Lineal + Armónicos)...")
def modelo_agua(t, m, c, a1, b1, a2, b2):
    w = 2 * np.pi / 12  # Frecuencia anual
    tendencia = m * t + c
    armonico_1 = a1 * np.cos(w * t) + b1 * np.sin(w * t)
    armonico_2 = a2 * np.cos(2 * w * t) + b2 * np.sin(2 * w * t)
    return tendencia + armonico_1 + armonico_2

# 3. CÁLCULO NUMÉRICO (Ajuste)
print("3. Ejecutando Optimizador (Mínimos Cuadrados)...")
p0_guess = [0, np.mean(consumo_real), 100000, 100000, 10000, 10000]

coeficientes_optimos, _ = curve_fit(
    modelo_agua, 
    meses_historicos, 
    consumo_real, 
    p0=p0_guess,
    maxfev=20000
)

# 4. MÉTRICAS DE VALIDACIÓN CIENTÍFICA
consumo_ajustado = modelo_agua(meses_historicos, *coeficientes_optimos)
r2 = r2_score(consumo_real, consumo_ajustado)

print("\n--- RESULTADOS ---")
print(f"Tendencia (m): {coeficientes_optimos[0]:.2f} Litros extra al mes")
print(f"Precisión del Modelo (R2): {r2:.4f}")

# 5. PREDICCIÓN Y PLOT
print("4. Dibujando el Espacio de Fases...")
meses_futuros = np.arange(len(consumo_real), len(consumo_real) + 12) # Proyectar 1 año
prediccion_futura = modelo_agua(meses_futuros, *coeficientes_optimos)

plt.figure(figsize=(14, 6))
plt.plot(meses_historicos, consumo_real, 'ko-', label='Datos Reales (AMAEM)', alpha=0.6, markersize=5)
plt.plot(meses_historicos, consumo_ajustado, 'b-', linewidth=2, label=f'Modelo Matemático ($R^2$ = {r2:.2f})')
plt.plot(meses_futuros, prediccion_futura, 'r--', linewidth=2.5, label='Predicción Próximos 12 meses')

plt.axvline(x=meses_historicos[-1], color='gray', linestyle=':', linewidth=2, label='Frontera Actual')
plt.title('Dinámica del Consumo de Agua en Alicante (Modelo de Fourier)', fontsize=16)
plt.ylabel('Consumo Total Mensual (Litros)', fontsize=12)

# Configurar etiquetas del eje X
ticks_pos = np.arange(0, len(consumo_real), max(1, len(consumo_real)//8))
plt.xticks(ticks_pos, [meses_fechas[i] for i in ticks_pos], rotation=45)

plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

**<center><span style="font-size:30px;">Bola de Cristal: Predicción y Cálculo de Error</span></center>**


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

In [ ]:
# =====================================================================
# 1. RECALCULAR EL ERROR (RMSE) PARA QUE NO FALLE
# =====================================================================
# Calculamos la predicción sobre el pasado para extraer el error medio (RMSE)
consumo_ajustado = modelo_agua(meses_historicos, *coeficientes_optimos)
rmse = np.sqrt(mean_squared_error(consumo_real, consumo_ajustado))

# =====================================================================
# 2. CONFIGURA AQUÍ LA FECHA QUE QUIERES PREDECIR
# =====================================================================
año_objetivo = 2027
mes_objetivo = 3  # 1 = Enero, 2 = Febrero, 3 = Marzo...

# =====================================================================
# 3. TRADUCCIÓN DEL TIEMPO Y PREDICCIÓN
# =====================================================================
# Detectar automáticamente cuándo empieza tu dataset (Enero 2022)
fecha_inicio = pd.to_datetime(meses_fechas[0]) 

# Calcular cuántos meses (índice 't') faltan hasta la fecha objetivo
t_objetivo = (año_objetivo - fecha_inicio.year) * 12 + (mes_objetivo - fecha_inicio.month)

# Predecir usando la ecuación matemática
consumo_predicho = modelo_agua(t_objetivo, *coeficientes_optimos)

# Calcular el margen de seguridad (95% de confianza = aprox 2 veces el RMSE)
margen_error_95 = 2 * rmse 

# =====================================================================
# 4. INFORME CORPORATIVO
# =====================================================================
meses_nombres = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", 
                 "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]

print(f"===========================================================")
print(f" 💧 PREDICCIÓN AMAEM: {meses_nombres[mes_objetivo-1]} de {año_objetivo}")
print(f"===========================================================")
print(f"Índice de tiempo del modelo (t): Mes {t_objetivo} desde el inicio.")
print(f"Consumo Base Esperado:      {consumo_predicho:,.0f} Litros")
print(f"Margen de Error (95% conf): ± {margen_error_95:,.0f} Litros")
print(f"-----------------------------------------------------------")
print(f"RANGO DE SEGURIDAD PARA OPERACIONES:")
print(f" > Mínimo garantizado:        {(consumo_predicho - margen_error_95):,.0f} Litros")
print(f" > Máximo posible (Pico):     {(consumo_predicho + margen_error_95):,.0f} Litros")
print(f"===========================================================")

In [ ]:
print(df.columns.tolist())

**<center><span style="font-size:30px;">Validación Científica: Train-Test Split (Backtesting)</span></center>**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

In [ ]:
# 1. PREPARACIÓN DE DATOS (Asumimos que df_ts y modelo_agua ya existen de la celda anterior)
# Vamos a esconder los últimos 12 meses (El "Futuro" para el modelo)
meses_a_esconder = 12
limite = len(consumo_real) - meses_a_esconder

# DATOS DE ENTRENAMIENTO (El Pasado que el modelo sí puede ver)
meses_train = meses_historicos[:limite]
consumo_train = consumo_real[:limite]
fechas_train = meses_fechas[:limite]

# DATOS DE PRUEBA (El "Futuro" escondido que usaremos para examinarle)
meses_test = meses_historicos[limite:]
consumo_test = consumo_real[limite:] # La realidad
fechas_test = meses_fechas[limite:]

# 2. ENTRENAMIENTO (Ajuste numérico SOLO con el pasado)
print(f"Entrenando el modelo ocultando los últimos {meses_a_esconder} meses...")
p0_guess = [0, np.mean(consumo_train), 100000, 100000, 10000, 10000]

coeficientes_ciegos, _ = curve_fit(
    modelo_agua, 
    meses_train, 
    consumo_train, 
    p0=p0_guess,
    maxfev=20000
)

# 3. EL EXAMEN FINAL (Predicción a ciegas)
consumo_train_ajustado = modelo_agua(meses_train, *coeficientes_ciegos) # Lo que aprendió
prediccion_ciega = modelo_agua(meses_test, *coeficientes_ciegos)         # Su adivinanza del futuro

# Calculamos el error SOLO en la zona que el modelo no conocía
r2_test = r2_score(consumo_test, prediccion_ciega)

print("\n--- RESULTADOS DEL EXAMEN (OUT-OF-SAMPLE) ---")
print(f"R2 en Entrenamiento (Datos vistos): {r2_score(consumo_train, consumo_train_ajustado):.4f}")
print(f"R2 en Prueba (Datos OCULTOS): {r2_test:.4f}")
print(f"Tendencia deducida a ciegas: {coeficientes_ciegos[0]:.2f} Litros/mes")

# 4. GRÁFICO REVELADOR
plt.figure(figsize=(14, 6))

# Zona de Entrenamiento (Pasado)
plt.plot(meses_train, consumo_train, 'ko-', label='Datos de Entrenamiento (Vistos)', alpha=0.5)
plt.plot(meses_train, consumo_train_ajustado, 'b-', linewidth=2, label='Ajuste de Entrenamiento')

# Zona de Prueba (El Examen)
plt.plot(meses_test, consumo_test, 'go-', markersize=8, label='LA REALIDAD OCULTA (Test)', alpha=0.8)
plt.plot(meses_test, prediccion_ciega, 'r--', linewidth=3, label=f'Predicción a Ciegas ($R^2$ = {r2_test:.2f})')

# Estética
plt.axvline(x=meses_train[-1] + 0.5, color='gray', linestyle='-', linewidth=2, label='Muro del Tiempo (Split)')
plt.axvspan(meses_train[-1] + 0.5, meses_historicos[-1], color='red', alpha=0.05, label='Zona de Predicción')

plt.title('Backtesting: ¿Sabe el modelo predecir datos que nunca ha visto?', fontsize=16)
plt.ylabel('Consumo Total Mensual (Litros)', fontsize=12)

# Eje X
ticks_pos = np.arange(0, len(consumo_real), max(1, len(consumo_real)//8))
plt.xticks(ticks_pos, [meses_fechas[i] for i in ticks_pos], rotation=45)

plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()